# 02 · Três métodos de seleção e votação de features

**Etapas do fluxo:** 4 (três métodos de importância/seleção) e 5 (votação → features eleitas do ciclo).

| Método | O que mede | Agregação para o atributo original |
|---|---|---|
| `SelectKBest(mutual_info_classif)` | dependência estatística (inclusive não linear) entre atributo e alvo | maior score entre suas colunas one-hot |
| Importância do `RandomForestClassifier` | redução de impureza nas árvores | soma das importâncias das colunas one-hot |
| `RFE` com `LogisticRegression` | ordem de eliminação recursiva do atributo menos útil | melhor posição entre suas colunas one-hot |

**Regra de votação (definida antes, em `config.yaml`):** cada método aprova seu *top-k*; é eleito o atributo com
pelo menos `votos_minimos` aprovações; se forem eleitos menos que `min_features`, completa-se pelos não eleitos com mais
votos e, em empate, menor posição média nos três rankings. Tudo é ajustado **somente no treino** do ciclo.

In [ ]:
import sys
import warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import matplotlib.pyplot as plt
import pandas as pd

from src.config import caminho, carregar_config

pd.set_option("display.max_columns", 50)
cfg = carregar_config(RAIZ)

In [ ]:
import mlflow
from src.ciclo import atualizar_contexto, carregar_contexto, diretorio_ciclo, obter_cycle_id, salvar_json
from src.features import executar_votacao
from src.rastreamento import run_etapa

cycle_id = obter_cycle_id(cfg)
contexto = carregar_contexto(cfg, cycle_id)
pasta_ciclo = diretorio_ciclo(cfg, cycle_id)
treino = pd.read_csv(caminho(cfg, cfg["dados"]["diretorio_processado"]) / "treino.csv")
print(cycle_id, contexto["data_version"], "| treino:", treino.shape)
print("Regra:", json.dumps({k: v for k, v in cfg["votacao"].items() if k != "metodos"}, ensure_ascii=False))

In [ ]:
resultado = executar_votacao(treino, cfg)
for metodo, ranking in resultado["resultados"].items():
    print(f"\n=== {metodo} (top {cfg['votacao']['top_k']}) ===")
    display(ranking.head(cfg["votacao"]["top_k"]).round(4))

## Tabela de votos e conjunto eleito

In [ ]:
tabela_votos = resultado["tabela_votos"]
eleitas = resultado["eleitas"]
display(tabela_votos)
print(f"Features eleitas ({len(eleitas)}):", eleitas)

cores = ["tab:green" if e else "tab:gray" for e in tabela_votos["eleita"]]
tabela_votos["total_votos"].iloc[::-1].plot.barh(color=cores[::-1], figsize=(8, 6), title="Votos por atributo (verde = eleito)")
plt.axvline(cfg["votacao"]["votos_minimos"] - 0.5, color="red", linestyle="--")
plt.show()

In [ ]:
tabela_votos.to_csv(pasta_ciclo / "votacao.csv")
for metodo, ranking in resultado["resultados"].items():
    ranking.to_csv(pasta_ciclo / f"ranking_{metodo}.csv")
salvar_json(pasta_ciclo / "features_eleitas.json", {"cycle_id": cycle_id, "data_version": contexto["data_version"], "features": eleitas})

with run_etapa(cfg, contexto, "votacao_features"):
    votacao = cfg["votacao"]
    mlflow.log_params({
        "metodo_votacao": f">= {votacao['votos_minimos']} de 3 votos no top-{votacao['top_k']}",
        "top_k": votacao["top_k"], "votos_minimos": votacao["votos_minimos"], "min_features": votacao["min_features"],
        "desempate": votacao["desempate"], "metodos": "selectkbest,random_forest,rfe",
        "features_eleitas": ",".join(eleitas),
    })
    mlflow.log_metrics({"n_features_eleitas": len(eleitas), "n_features_candidatas": len(tabela_votos), "n_colunas_transformadas": resultado["n_colunas_transformadas"]})
    for arquivo in ["votacao.csv", "features_eleitas.json", *[f"ranking_{m}.csv" for m in resultado["resultados"]]]:
        mlflow.log_artifact(str(pasta_ciclo / arquivo))

contexto = atualizar_contexto(cfg, cycle_id, features_eleitas=eleitas)